<a href="https://colab.research.google.com/github/Bpatnaik470/Bpatnaik470/blob/main/Mercedes_Benz_Greener_Manufacturing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import zipfile
import pandas as pd

# Unzip train.zip
with zipfile.ZipFile('/content/train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/train')

# Unzip test.zip
with zipfile.ZipFile('/content/test.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/test')

import os

train_csv_path = None
test_csv_path = None

for root, dirs, files in os.walk('/content/train'):
    for file in files:
        if file.endswith('.csv'):
            train_csv_path = os.path.join(root, file)
            break
    if train_csv_path: break

for root, dirs, files in os.walk('/content/test'):
    for file in files:
        if file.endswith('.csv'):
            test_csv_path = os.path.join(root, file)
            break
    if test_csv_path: break

if train_csv_path and test_csv_path:
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    print(f"Train data loaded from: {train_csv_path}")
    print(f"Test data loaded from: {test_csv_path}")
    print("Train DataFrame head:")
    display(train_df.head())
    print("Test DataFrame head:")
    display(test_df.head())
else:
    print("Could not find train.csv or test.csv after unzipping.")
    print("Contents of /content/train:", os.listdir('/content/train'))
    print("Contents of /content/test:", os.listdir('/content/test'))


Train data loaded from: /content/train/train.csv
Test data loaded from: /content/test/test.csv
Train DataFrame head:


,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,a,d,u,j,o,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,e,d,y,l,o,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,c,d,x,j,x,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,az,t,n,f,d,x,l,e,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,az,v,n,f,d,h,d,n,...,0,0,0,0,0,0,0,0,0,0


Test DataFrame head:


,ID,X0,X1,X2,X3,X4,X5,X6,X8,X10,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,1,az,v,n,f,d,t,a,w,0,...,0,0,0,1,0,0,0,0,0,0
1,2,t,b,ai,a,d,b,g,y,0,...,0,0,1,0,0,0,0,0,0,0
2,3,az,v,as,f,d,a,j,j,0,...,0,0,0,1,0,0,0,0,0,0
3,4,az,l,n,f,d,z,l,n,0,...,0,0,0,1,0,0,0,0,0,0
4,5,w,s,as,c,d,y,i,m,0,...,1,0,0,0,0,0,0,0,0,0


### Removing columns with zero variance

Now, I will remove any column(s) from both the train and test datasets where the variance is equal to zero. These columns do not provide any useful information for modeling.

In [2]:
from sklearn.feature_selection import VarianceThreshold

# Identify columns with zero variance in train_df
selector = VarianceThreshold(threshold=0)
selector.fit(train_df.select_dtypes(include=['number'])) # Only fit on numeric columns

# Get names of columns with non-zero variance
non_zero_variance_cols_train = train_df.select_dtypes(include=['number']).columns[selector.get_support()]

# Identify columns with zero variance in test_df
selector_test = VarianceThreshold(threshold=0)
selector_test.fit(test_df.select_dtypes(include=['number'])) # Only fit on numeric columns

# Get names of columns with non-zero variance
non_zero_variance_cols_test = test_df.select_dtypes(include=['number']).columns[selector_test.get_support()]

# Combine and get unique columns to keep, ensuring we only drop if zero variance in both, or handle separately if desired.
# For now, let's keep only columns that have non-zero variance in both, or drop if zero in respective dataframe.

# Columns to drop from train_df
zero_variance_cols_train = train_df.select_dtypes(include=['number']).columns[~selector.get_support()]
print(f"Columns with zero variance in train_df: {list(zero_variance_cols_train)}")
train_df = train_df.drop(columns=zero_variance_cols_train, errors='ignore')

# Columns to drop from test_df
zero_variance_cols_test = test_df.select_dtypes(include=['number']).columns[~selector_test.get_support()]
print(f"Columns with zero variance in test_df: {list(zero_variance_cols_test)}")
test_df = test_df.drop(columns=zero_variance_cols_test, errors='ignore')

print("Train DataFrame shape after removing zero variance columns:", train_df.shape)
print("Test DataFrame shape after removing zero variance columns:", test_df.shape)


Columns with zero variance in train_df: ['X11', 'X93', 'X107', 'X233', 'X235', 'X268', 'X289', 'X290', 'X293', 'X297', 'X330', 'X347']
Columns with zero variance in test_df: ['X257', 'X258', 'X295', 'X296', 'X369']
Train DataFrame shape after removing zero variance columns: (4209, 366)
Test DataFrame shape after removing zero variance columns: (4209, 372)


### Checking for null and unique values

Next, I will check for null values and the number of unique values for each column in both the training and test datasets. This helps in understanding data quality and identifying categorical columns.

In [3]:
print("\n--- Train DataFrame Null Values ---")
display(train_df.isnull().sum()[train_df.isnull().sum() > 0].sort_values(ascending=False))

print("\n--- Test DataFrame Null Values ---")
display(test_df.isnull().sum()[test_df.isnull().sum() > 0].sort_values(ascending=False))

print("\n--- Train DataFrame Unique Values ---")
for col in train_df.columns:
    if train_df[col].nunique() < 20: # Display unique values for columns with less than 20 unique values
        print(f"Column '{col}': {train_df[col].nunique()} unique values: {train_df[col].unique()}")
    else:
        print(f"Column '{col}': {train_df[col].nunique()} unique values (too many to list)")

print("\n--- Test DataFrame Unique Values ---")
for col in test_df.columns:
    if test_df[col].nunique() < 20: # Display unique values for columns with less than 20 unique values
        print(f"Column '{col}': {test_df[col].nunique()} unique values: {test_df[col].unique()}")
    else:
        print(f"Column '{col}': {test_df[col].nunique()} unique values (too many to list)")



--- Train DataFrame Null Values ---


,0



--- Test DataFrame Null Values ---


,0



--- Train DataFrame Unique Values ---
Column 'ID': 4209 unique values (too many to list)
Column 'y': 2545 unique values (too many to list)
Column 'X0': 47 unique values (too many to list)
Column 'X1': 27 unique values (too many to list)
Column 'X2': 44 unique values (too many to list)
Column 'X3': 7 unique values: ['a' 'e' 'c' 'f' 'd' 'b' 'g']
Column 'X4': 4 unique values: ['d' 'b' 'c' 'a']
Column 'X5': 29 unique values (too many to list)
Column 'X6': 12 unique values: ['j' 'l' 'd' 'h' 'i' 'a' 'g' 'c' 'k' 'e' 'f' 'b']
Column 'X8': 25 unique values (too many to list)
Column 'X10': 2 unique values: [0 1]
Column 'X12': 2 unique values: [0 1]
Column 'X13': 2 unique values: [1 0]
Column 'X14': 2 unique values: [0 1]
Column 'X15': 2 unique values: [0 1]
Column 'X16': 2 unique values: [0 1]
Column 'X17': 2 unique values: [0 1]
Column 'X18': 2 unique values: [1 0]
Column 'X19': 2 unique values: [0 1]
Column 'X20': 2 unique values: [0 1]
Column 'X21': 2 unique values: [1 0]
Column 'X22': 2 uni

### Applying Label Encoder

Now, I will apply `LabelEncoder` to all categorical columns in both the training and test datasets. This converts categorical labels into numerical format, which is required for most machine learning algorithms.

In [4]:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns (object type)
categorical_cols_train = train_df.select_dtypes(include=['object']).columns
categorical_cols_test = test_df.select_dtypes(include=['object']).columns

# Ensure we apply encoder to columns present in both, and handle discrepancies if any.
# For simplicity, apply to each dataframe independently for now.

for col in categorical_cols_train:
    le = LabelEncoder()
    # Handle potential NaNs before encoding if not handled already. Fill with a placeholder.
    train_df[col] = train_df[col].fillna('Missing').astype(str)
    train_df[col] = le.fit_transform(train_df[col])
    print(f"Label encoded column (train): {col}")

for col in categorical_cols_test:
    le = LabelEncoder()
    # Handle potential NaNs before encoding. Fill with a placeholder.
    test_df[col] = test_df[col].fillna('Missing').astype(str)
    test_df[col] = le.fit_transform(test_df[col])
    print(f"Label encoded column (test): {col}")

print("Train DataFrame head after label encoding:")
display(train_df.head())
print("Test DataFrame head after label encoding:")
display(test_df.head())


Label encoded column (train): X0
Label encoded column (train): X1
Label encoded column (train): X2
Label encoded column (train): X3
Label encoded column (train): X4
Label encoded column (train): X5
Label encoded column (train): X6
Label encoded column (train): X8
Label encoded column (test): X0
Label encoded column (test): X1
Label encoded column (test): X2
Label encoded column (test): X3
Label encoded column (test): X4
Label encoded column (test): X5
Label encoded column (test): X6
Label encoded column (test): X8
Train DataFrame head after label encoding:


,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,32,23,17,0,3,24,9,14,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,32,21,19,4,3,28,11,14,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,20,24,34,2,3,27,9,23,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,20,21,34,5,3,27,11,4,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,20,23,34,5,3,12,3,13,...,0,0,0,0,0,0,0,0,0,0


Test DataFrame head after label encoding:


,ID,X0,X1,X2,X3,X4,X5,X6,X8,X10,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,1,21,23,34,5,3,26,0,22,0,...,0,0,0,1,0,0,0,0,0,0
1,2,42,3,8,0,3,9,6,24,0,...,0,0,1,0,0,0,0,0,0,0
2,3,21,23,17,5,3,0,9,9,0,...,0,0,0,1,0,0,0,0,0,0
3,4,21,13,34,5,3,31,11,13,0,...,0,0,0,1,0,0,0,0,0,0
4,5,45,20,17,2,3,30,8,12,0,...,1,0,0,0,0,0,0,0,0,0


### Performing Dimensionality Reduction (PCA)

I will now perform dimensionality reduction using Principal Component Analysis (PCA) to reduce the number of features while retaining most of the variance in the data. This can help in reducing computational cost and improving model performance. First, we need to handle any remaining NaNs and scale the data.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# Impute any remaining NaN values with the mean for numerical columns before scaling and PCA
for col in train_df.select_dtypes(include=np.number).columns:
    if train_df[col].isnull().any():
        train_df[col] = train_df[col].fillna(train_df[col].mean())

for col in test_df.select_dtypes(include=np.number).columns:
    if test_df[col].isnull().any():
        test_df[col] = test_df[col].fillna(test_df[col].mean())

# Separate target variable from train_df if it exists, assuming the last column is the target for now.
# If the target column is known, replace 'target_column_name' below.
# For this example, let's assume there's a column 'target' in train_df that needs to be predicted.
# If there's no explicit target, this step will need adjustment.

# Find a suitable target column. Often it's the last column or named 'target', 'label', etc.
# This is a generic approach; you might need to adjust 'target_column_name' based on your dataset.
# Let's assume the target column is not present in test_df (which is standard for prediction tasks)
# and that it's a numeric column in train_df.

# First, identify common columns between train_df and test_df for feature set.
common_cols = list(set(train_df.columns) & set(test_df.columns))

# The target column must be in train_df but not in test_df (or explicitly excluded from test_df).
# Let's try to infer a target column: a column in train_df not present in test_df and is numeric.
possible_targets = [col for col in train_df.columns if col not in test_df.columns and pd.api.types.is_numeric_dtype(train_df[col])]

if possible_targets:
    target_column_name = possible_targets[0] # Pick the first one if multiple found
    print(f"Inferred target column: {target_column_name}")
    X_train = train_df.drop(columns=[target_column_name, 'id'] if 'id' in train_df.columns else [target_column_name]) # Assuming 'id' is not a feature
    y_train = train_df[target_column_name]
else:
    print("Could not infer a clear target column. Proceeding without separating target for now.")
    print("Please manually define your target column if this is incorrect.")
    X_train = train_df.drop(columns=['id'] if 'id' in train_df.columns else []) # Assume 'id' is not a feature
    y_train = None # No target identified

X_test = test_df.drop(columns=['id'] if 'id' in test_df.columns else []) # Assuming 'id' is not a feature

# Align columns between X_train and X_test after potential target removal
# This is crucial if some columns were dropped due to zero variance in only one dataset
missing_in_test = set(X_train.columns) - set(X_test.columns)
for col in missing_in_test:
    X_test[col] = 0 # Or other appropriate imputation like mean/median of X_train[col]

missing_in_train = set(X_test.columns) - set(X_train.columns)
for col in missing_in_train:
    X_train[col] = 0 # Or other appropriate imputation like mean/median of X_test[col]

X_test = X_test[X_train.columns] # Ensure order of columns is the same


# Scale the data before applying PCA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA. Let's aim for 95% variance explained or a fixed number of components.
# For now, let's target a reasonable number of components, say 0.95 explained variance ratio.
pca = PCA(n_components=0.95) # Retain components explaining 95% of the variance

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original number of features: {X_train.shape[1]}")
print(f"Number of features after PCA: {X_train_pca.shape[1]}")
print(f"Explained variance ratio by PCA components: {pca.explained_variance_ratio_.sum():.4f}")

print("Shape of X_train_pca:", X_train_pca.shape)
print("Shape of X_test_pca:", X_test_pca.shape)

# Convert PCA results back to DataFrame for consistency if needed, or keep as numpy array for XGBoost
# X_train_pca_df = pd.DataFrame(X_train_pca, index=X_train.index)
# X_test_pca_df = pd.DataFrame(X_test_pca, index=X_test.index)


Inferred target column: y
Original number of features: 377
Number of features after PCA: 149
Explained variance ratio by PCA components: 0.9508
Shape of X_train_pca: (4209, 149)
Shape of X_test_pca: (4209, 149)


### Predicting test_df values using XGBoost

Finally, I will train an XGBoost model using the processed training data and predict the values for the test dataset. This step assumes a regression or classification task based on the target variable identified earlier. I will use a regressor if `y_train` is numerical and continuous, otherwise, a classifier.

In [6]:
import xgboost as xgb

# Determine if the task is regression or classification based on the target variable
if y_train is not None and pd.api.types.is_numeric_dtype(y_train):
    if y_train.nunique() > 20: # Heuristic: if more than 20 unique values, assume regression
        print("Target variable appears to be continuous (regression task).")
        model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
    else: # Otherwise, classification
        print("Target variable appears to be discrete (classification task).")
        # For classification, adjust objective based on number of classes
        if y_train.nunique() == 2:
            model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100, random_state=42)
        else:
            model = xgb.XGBClassifier(objective='multi:softmax', num_class=y_train.nunique(), n_estimators=100, random_state=42)

    # Train the model
    model.fit(X_train_pca, y_train)

    # Make predictions on the PCA-transformed test data
    predictions = model.predict(X_test_pca)

    print("\nPredictions on test data using XGBoost:")
    display(pd.Series(predictions).head())

    # If 'id' column exists in original test_df, create a submission-like DataFrame
    if 'id' in test_df.columns:
        submission_df = pd.DataFrame({'id': test_df['id'], 'predicted_value': predictions})
        print("\nSample Submission DataFrame:")
        display(submission_df.head())
    else:
        print("No 'id' column found in test_df to create a submission file.")

elif y_train is None:
    print("Cannot perform prediction as the target variable (y_train) was not identified or provided.")
    print("Please ensure a target column is specified or inferable from your data.")
else:
    print("Target variable is not numeric. Cannot proceed with standard XGBoost without further context.")


Target variable appears to be continuous (regression task).

Predictions on test data using XGBoost:


,0
0,79.621506
1,108.141594
2,92.376732
3,82.569916
4,103.765457


No 'id' column found in test_df to create a submission file.
